In [1]:
# making corpus or words from comments
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
#from keras.utils import generic_utils
from keras.callbacks import History
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from keras.callbacks import EarlyStopping
from keras.layers import Dropout
import re
#from tensorflow.keras.utils import to_categorical
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import confusion_matrix,accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import precision_score, recall_score, f1_score

history =History()
dataset = pd.read_csv('/content/new.csv', encoding='cp437')
dataset['Comment']=dataset['Comment'].astype(str)




# The maximum number of words to be used. (most frequent)
MAX_NB_WORDS = 50000
# Max number of words in each complaint.
MAX_SEQUENCE_LENGTH = 250
# This is fixed.
EMBEDDING_DIM = 100


tokenizer = Tokenizer(num_words=MAX_NB_WORDS, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~', lower=True)
tokenizer.fit_on_texts(dataset['Comment'].values)

X=tokenizer.texts_to_sequences(dataset['Comment'].values)

X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH)
print('Shape of data tensor:', X.shape)


Y = pd.get_dummies(dataset['sentiment']).values
print('Shape of label tensor:', Y.shape)
label_mapping = pd.get_dummies(dataset['sentiment']).columns.tolist()
print(label_mapping)
seed = 35

X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.30, random_state=42)#np.random.seed(seed))
print(X_train.shape,Y_train.shape)
print(X_test.shape,Y_test.shape)

model = Sequential()
model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
#Adding a dense hidden layer
model.add(Dense(3, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(3, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])#optimizer=Adam(learning_rate=1e-4)
print(model.summary())
epochs = 20
batch_size=64
history = model.fit(X_train, Y_train,epochs=epochs,batch_size=batch_size,validation_split=0.2,callbacks= None)
accr = model.evaluate(X_test,Y_test)
print("Accuracy:", accr[1])
#print('Test set\n  Loss: {:0.3f}\n  Accuracy: {:0.3f}'.format(accr[0],accr[1]))

#rounded_predictions = model.predict(X_test)#, batch_size=128, verbose=1)
rounded_predictions = (model.predict(X_test) > 0.5).astype("int32")
rounded_labels=np.argmax(Y_test, axis=1)


from sklearn.metrics import confusion_matrix,precision_recall_fscore_support
#cm = confusion_matrix(rounded_labels, rounded_predictions)
Y_test_arg=np.argmax(Y_test,axis=1)
Y_pred = np.argmax(model.predict(X_test),axis=1)
print('Confusion Matrix')
cm=confusion_matrix(Y_test_arg, Y_pred)
print(cm)
class_accuracy=[]
for ci in range(len(cm)):
    # True positives for the current class
    TP = cm[ci, ci]

    # Total instances for the current class (denominator)
    total_instances = sum(cm[ci, :])

    # Calculate class accuracy and append to the list
    accuracy = TP / total_instances if total_instances > 0 else 0.0
    class_accuracy.append(accuracy)
# Print the accuracy per class

for ci, accuracy in enumerate(class_accuracy, start=1):
  print(f"Class {ci-1} Accuracy: {accuracy:.2%}")
from sklearn.metrics import classification_report

# Generate a classification report
report = classification_report(Y_test_arg, Y_pred)
print(report)

#for ci in range(len(precision)):
 # print(f"Class{ci}-Precision:{precision[ci]},Recall: {recall[ci]},F1-score:{f1_score[ci]}")



#tp_and_fn = cm.sum(1)
#tp_and_fp = cm.sum(0)
#tp = cm.diagonal()

#precision = tp / tp_and_fp
#recall = tp / tp_and_fn

#print(precision)
#print(recall)

#precision= cm[0][0]/(cm[0][0]+cm[1][0])

#print( 'Precision: {:0.3f}'.format(precision))
#recall= cm[0][0]/(cm[0][0]+cm[0][1])
#print('recall: {:0.3f}'.format(recall))
#fscore=(2*cm[0][0])/(2*cm[0][0]+cm[0][1]+cm[1][0])
#print('fscore: {:0.3f}'.format(fscore))

Shape of data tensor: (20229, 250)
Shape of label tensor: (20229, 3)
['Negative', 'Neutral', 'Positive']
(14160, 250) (14160, 3)
(6069, 250) (6069, 3)
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 250, 100)          5000000   
                                                                 
 spatial_dropout1d (Spatial  (None, 250, 100)          0         
 Dropout1D)                                                      
                                                                 
 lstm (LSTM)                 (None, 100)               80400     
                                                                 
 dense (Dense)               (None, 3)                 303       
                                                                 
 dropout (Dropout)           (None, 3)                 0         
                                     

In [2]:
# prompt: change the above code to work better for ternary classification

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from keras.callbacks import History
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from keras.callbacks import EarlyStopping
from keras.layers import Dropout
import re
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import confusion_matrix,accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix,precision_recall_fscore_support
#
#
#
# making corpus or words from comments
#from keras.utils import generic_utils
#from tensorflow.keras.utils import to_categorical

history =History()
dataset = pd.read_csv('/content/new.csv', encoding='cp437')
dataset['Comment']=dataset['Comment'].astype(str)




# The maximum number of words to be used. (most frequent)
MAX_NB_WORDS = 50000
# Max number of words in each complaint.
MAX_SEQUENCE_LENGTH = 250
# This is fixed.
EMBEDDING_DIM = 100


tokenizer = Tokenizer(num_words=MAX_NB_WORDS, filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~', lower=True)
tokenizer.fit_on_texts(dataset['Comment'].values)

X=tokenizer.texts_to_sequences(dataset['Comment'].values)

X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH)
print('Shape of data tensor:', X.shape)


Y = pd.get_dummies(dataset['sentiment']).values
print('Shape of label tensor:', Y.shape)
label_mapping = pd.get_dummies(dataset['sentiment']).columns.tolist()
print(label_mapping)
seed = 35

X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.30, random_state=42)#np.random.seed(seed))
print(X_train.shape,Y_train.shape)
print(X_test.shape,Y_test.shape)

model = Sequential()
model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
#Adding a dense hidden layer
# model.add(Dense(3, activation='relu'))
# model.add(Dropout(0.2))
model.add(Dense(3, activation='softmax')) # Output layer for ternary classification
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])#optimizer=Adam(learning_rate=1e-4)
print(model.summary())
epochs = 20
batch_size=64
history = model.fit(X_train, Y_train,epochs=epochs,batch_size=batch_size,validation_split=0.2,callbacks= None)
accr = model.evaluate(X_test,Y_test)
print("Accuracy:", accr[1])
#print('Test set\n  Loss: {:0.3f}\n  Accuracy: {:0.3f}'.format(accr[0],accr[1]))

#rounded_predictions = model.predict(X_test)#, batch_size=128, verbose=1)
rounded_predictions = (model.predict(X_test) > 0.5).astype("int32")
rounded_labels=np.argmax(Y_test, axis=1)


#cm = confusion_matrix(rounded_labels, rounded_predictions)
Y_test_arg=np.argmax(Y_test,axis=1)
Y_pred = np.argmax(model.predict(X_test),axis=1)
print('Confusion Matrix')
cm=confusion_matrix(Y_test_arg, Y_pred)
print(cm)
class_accuracy=[]
for ci in range(len(cm)):
    # True positives for the current class
    TP = cm[ci, ci]

    # Total instances for the current class (denominator)
    total_instances = sum(cm[ci, :])

    # Calculate class accuracy and append to the list
    accuracy = TP / total_instances if total_instances > 0 else 0.0
    class_accuracy.append(accuracy)
# Print the accuracy per class

for ci, accuracy in enumerate(class_accuracy, start=0): # Start enumeration from 0 for class labels
  print(f"Class {ci} Accuracy: {accuracy:.2%}")

# Generate a classification report
report = classification_report(Y_test_arg, Y_pred)
print(report)


Shape of data tensor: (20229, 250)
Shape of label tensor: (20229, 3)
['Negative', 'Neutral', 'Positive']
(14160, 250) (14160, 3)
(6069, 250) (6069, 3)
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 250, 100)          5000000   
                                                                 
 spatial_dropout1d_1 (Spati  (None, 250, 100)          0         
 alDropout1D)                                                    
                                                                 
 lstm_1 (LSTM)               (None, 100)               80400     
                                                                 
 dense_2 (Dense)             (None, 3)                 303       
                                                                 
Total params: 5080703 (19.38 MB)
Trainable params: 5080703 (19.38 MB)
Non-trainable params: 0 (0.00 B

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
